In [1]:
!pip install localtileserver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 3.9 MB/s eta 0:00:00


In [2]:
# ---------------------------------------------------------
# PHASE 1: PRECISION DATA PREPARATION
# ---------------------------------------------------------
import ee
# ---------------------------------------------------------
# 1. Initialize Earth Engine
# ---------------------------------------------------------

print("Initializing Earth Engine...")

try:
    ee.Initialize(project='replicating-paper')
    print("Earth Engine initialized successfully.")
except Exception as e:
    print("Authentication required. Starting authentication flow...")
    ee.Authenticate()
    ee.Initialize(project='replicating-paper')
    print("Earth Engine initialized after authentication.")


# ---------------------------------------------------------
# 2. Define Region of Interest (ROI)
# ---------------------------------------------------------

print("Defining Region of Interest (Sanjay Van)...")

roi = ee.Geometry.Rectangle([77.17, 28.52, 77.18, 28.54])
print("ROI defined.")


# ---------------------------------------------------------
# 3. Sentinel-2 Cloud & Shadow Mask Function
# ---------------------------------------------------------

def mask_s2_clouds(image):
    """
    Removes clouds, shadows, cirrus, saturated pixels,
    and dilates the mask to remove cloud-edge contamination.
    """

    image = ee.Image(image)

    scl = image.select('SCL')

    # SCL classes to remove
    mask = (
        scl.eq(1)   # saturated / defective
        .Or(scl.eq(3))   # cloud shadow
        .Or(scl.eq(7))   # unclassified
        .Or(scl.eq(8))   # cloud medium probability
        .Or(scl.eq(9))   # cloud high probability
        .Or(scl.eq(10))  # cirrus
    )

    # Dilate mask to remove spectral halos
    dilated = mask.focalMax(radius=60, units='meters')

    clean_mask = dilated.Not()

    return (
        image
        .updateMask(clean_mask)
        .divide(10000)          # scale reflectance
        .clip(roi)
        .copyProperties(image, image.propertyNames())
    )


print("Cloud masking function ready.")


# ---------------------------------------------------------
# 4. Load Sentinel-2 Collection (2024)
# ---------------------------------------------------------

print("Loading Sentinel-2 SR collection...")

s2_raw = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(roi)
    .filterDate('2024-01-01', '2025-01-01')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
)

print("Sentinel-2 collection loaded.")


# ---------------------------------------------------------
# 5. Apply Cloud Mask
# ---------------------------------------------------------

print("Applying cloud masking...")

s2_col = s2_raw.map(mask_s2_clouds)

print("Cloud masking applied.")


# ---------------------------------------------------------
# 6. Generate Annual Median Composite
# ---------------------------------------------------------

print("Generating 2024 median composite...")

s2_median = s2_col.median().clip(roi)

print("Median composite created.")


# ---------------------------------------------------------
# 7. Define Master Projection
# ---------------------------------------------------------

print("Extracting master projection...")

master_projection = s2_raw.first().select('B2').projection()

print("Master projection locked to Sentinel-2 10m grid.")


# ---------------------------------------------------------
# 8. Debug Information
# ---------------------------------------------------------

print("Running collection diagnostics...")

try:
    collection_size = s2_raw.size().getInfo()
    print("Total Sentinel-2 images found:", collection_size)
except Exception as e:
    print("Collection size check skipped (server-side evaluation).")


print("--------------------------------------------------")
print("PHASE 1 COMPLETE")
print("Clean Sentinel-2 collection + median composite ready.")
print("--------------------------------------------------")

Initializing Earth Engine...
Authentication required. Starting authentication flow...
Earth Engine initialized after authentication.
Defining Region of Interest (Sanjay Van)...
ROI defined.
Cloud masking function ready.
Loading Sentinel-2 SR collection...
Sentinel-2 collection loaded.
Applying cloud masking...
Cloud masking applied.
Generating 2024 median composite...
Median composite created.
Extracting master projection...
Master projection locked to Sentinel-2 10m grid.
Running collection diagnostics...
Total Sentinel-2 images found: 38
--------------------------------------------------
PHASE 1 COMPLETE
Clean Sentinel-2 collection + median composite ready.
--------------------------------------------------


In [3]:
import rasterio
from pyproj import Transformer
import ee

# 1. Load the TIF to get its native CRS (Coordinate Reference System)
tif_path = '/content/drive/MyDrive/GeoSpatial_Data_for_Mapping/Spot_1_10-08-25.tif'

with rasterio.open(tif_path) as src:
    # Get the drone's CRS (usually EPSG:32643 for Delhi UTM)
    drone_crs = src.crs
    left, bottom, right, top = src.bounds

# 2. Setup the Transformer (From Drone UTM to Lat/Lon)
# 'always_xy=True' ensures we get (Long, Lat) which GEE expects
transformer = Transformer.from_crs(drone_crs, "EPSG:4326", always_xy=True)

# 3. Transform the Corners
lon_min, lat_min = transformer.transform(left, bottom)
lon_max, lat_max = transformer.transform(right, top)

# 4. Create the NEW ROI for your Pipeline
# This "Spot ROI" will be much smaller than your Sanjay Van ROI
spot_roi = ee.Geometry.Rectangle([lon_min, lat_min, lon_max, lat_max])

print(f"--- COORDINATES ---")
print(f"Lat/Lon Min: {lat_min}, {lon_min}")
print(f"Lat/Lon Max: {lat_max}, {lon_max}")

# Now, you can set your pipeline to use this new ROI
roi = spot_roi

--- COORDINATES ---
Lat/Lon Min: 28.531656278420357, 77.17553822432453
Lat/Lon Max: 28.533993879350852, 77.177920206531


In [4]:
# ---------------------------------------------------------
# PHASE 2: DNA FEATURE ENGINEERING
# ---------------------------------------------------------

print("Starting Phase 2: DNA Feature Engineering...")


# ---------------------------------------------------------
# 2.1 Phenological DNA (NDVI Series)
# ---------------------------------------------------------

print("Calculating NDVI time series...")

def add_ndvi(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return img.addBands(ndvi)

ndvi_col = s2_col.map(add_ndvi).select('NDVI')

# Productivity = mean NDVI
productivity = ndvi_col.mean().rename('Productivity')

# Seasonality = NDVI stdDev
seasonality = ndvi_col.reduce(ee.Reducer.stdDev()).rename('Seasonality')

print("Phenological DNA extracted.")


# ---------------------------------------------------------
# 2.2 Spectral DNA (Tasseled Cap Wetness)
# ---------------------------------------------------------

print("Computing Tasseled Cap indices...")

def apply_tasseled_cap(image):

    B2 = image.select('B2')
    B3 = image.select('B3')
    B4 = image.select('B4')
    B8 = image.select('B8')
    B11 = image.select('B11')
    B12 = image.select('B12')

    # Brightness
    tcb = (
        B2.multiply(0.3029)
        .add(B3.multiply(0.2758))
        .add(B4.multiply(0.2334))
        .add(B8.multiply(0.5037))
        .add(B11.multiply(0.4782))
        .add(B12.multiply(0.3156))
        .rename('TCB')
    )

    # Wetness
    tcw = (
        B2.multiply(0.1509)
        .add(B3.multiply(0.1973))
        .add(B4.multiply(0.3283))
        .add(B8.multiply(0.3407))
        .add(B11.multiply(-0.7117))
        .add(B12.multiply(-0.4559))
        .rename('TCW')
    )

    return ee.Image.cat([tcb, tcw])


tc_indices = apply_tasseled_cap(s2_median)

print("Tasseled Cap indices computed.")


# ---------------------------------------------------------
# 2.3 Built-up Index (NDBI)
# ---------------------------------------------------------

print("Calculating NDBI...")

ndbi = s2_median.normalizedDifference(['B11', 'B8']).rename('NDBI')

print("NDBI calculated.")


# ---------------------------------------------------------
# 2.4 Structural DNA (Sentinel-1 Radar VH)
# ---------------------------------------------------------

print("Loading Sentinel-1 collection...")

s1_vh = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(roi)
    .filterDate('2024-01-01', '2025-01-01')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .filter(ee.Filter.eq('resolution_meters', 10))
    .select('VH')
    .median()
    .focalMean(radius=30, units='meters')
    .clip(roi)
    .rename('Radar_VH')
)

print("Radar VH extracted.")


# ---------------------------------------------------------
# 2.4.b Topographic DNA (NASADEM Elevation & Slope)
# ---------------------------------------------------------

print("Loading Topographic data...")

# Load NASADEM (30m resolution global elevation)
elevation = ee.Image("NASA/NASADEM_HGT/001").select('elevation').clip(roi).rename('Elevation')

# Calculate Slope (in degrees) from Elevation
slope = ee.Terrain.slope(elevation).rename('Slope')

print("Elevation and Slope extracted.")


# ---------------------------------------------------------
# 2.5 Assemble DNA Feature Stack (UPGRADED)
# ---------------------------------------------------------

print("Assembling Universal DNA feature stack...")

dna_stack_raw = ee.Image.cat([
    productivity,
    seasonality,
    tc_indices,
    ndbi,
    s1_vh,
    elevation, # <-- Added dynamically!
    slope      # <-- Added dynamically!
])

print("---------------------------------------------------")
print("PHASE 2 COMPLETE")
print("Bands in Universal DNA Stack:")
print(dna_stack_raw.bandNames().getInfo())
print("---------------------------------------------------")

Starting Phase 2: DNA Feature Engineering...
Calculating NDVI time series...
Phenological DNA extracted.
Computing Tasseled Cap indices...
Tasseled Cap indices computed.
Calculating NDBI...
NDBI calculated.
Loading Sentinel-1 collection...
Radar VH extracted.
Loading Topographic data...
Elevation and Slope extracted.
Assembling Universal DNA feature stack...
---------------------------------------------------
PHASE 2 COMPLETE
Bands in Universal DNA Stack:
['Productivity', 'Seasonality', 'TCB', 'TCW', 'NDBI', 'Radar_VH', 'Elevation', 'Slope']
---------------------------------------------------


In [5]:
# ---------------------------------------------------------
# PHASE 3: GEOMORPHIC PURIFICATION (MASKING)
# ---------------------------------------------------------

print("Starting Phase 3: Geomorphic Purification...")


# ---------------------------------------------------------
# 3.1 Water Detection (MNDWI)
# ---------------------------------------------------------

print("Detecting water bodies...")

mndwi = s2_median.normalizedDifference(['B3', 'B11']).rename('MNDWI')

water_mask = mndwi.gt(0.1)   # pixels likely water

print("Water mask generated.")


# ---------------------------------------------------------
# 3.2 Built-up / Urban Detection (NDBI)
# ---------------------------------------------------------

print("Detecting urban/built-up areas...")

# Use NDBI already computed in Phase 2
ndbi = dna_stack_raw.select('NDBI')

urban_mask = ndbi.gt(0.1)

print("Urban mask generated.")


# ---------------------------------------------------------
# 3.3 Barren / Degraded Land Detection
# ---------------------------------------------------------

print("Detecting sparse vegetation...")

# Use productivity (mean NDVI)
barren_mask = productivity.lt(0.25)

print("Barren land mask generated.")


# ---------------------------------------------------------
# 3.4 Combine Masks (Non-Forest Detection)
# ---------------------------------------------------------

print("Combining masks into non-forest logic...")

non_forest_mask = (
    water_mask
    .Or(urban_mask)
    .Or(barren_mask)
)

print("Non-forest mask created.")


# ---------------------------------------------------------
# 3.5 Mask Erosion (Remove Edge Pixels)
# ---------------------------------------------------------

print("Applying mask erosion to remove edge contamination...")

forest_only_domain = (
    non_forest_mask
    .Not()
    .focalMin(radius=15, units='meters')
    .rename('Forest_Domain')
)

print("Forest-only domain generated.")


# ---------------------------------------------------------
# 3.6 Apply Mask to DNA Stack
# ---------------------------------------------------------

print("Applying forest mask to DNA stack...")

dna_stack_purified = dna_stack_raw.updateMask(forest_only_domain)

print("---------------------------------------------------")
print("PHASE 3 COMPLETE")
print("Non-forest areas removed and mask erosion applied.")
print("Bands remaining:")
print(dna_stack_purified.bandNames().getInfo())
print("---------------------------------------------------")

Starting Phase 3: Geomorphic Purification...
Detecting water bodies...
Water mask generated.
Detecting urban/built-up areas...
Urban mask generated.
Detecting sparse vegetation...
Barren land mask generated.
Combining masks into non-forest logic...
Non-forest mask created.
Applying mask erosion to remove edge contamination...
Forest-only domain generated.
Applying forest mask to DNA stack...
---------------------------------------------------
PHASE 3 COMPLETE
Non-forest areas removed and mask erosion applied.
Bands remaining:
['Productivity', 'Seasonality', 'TCB', 'TCW', 'NDBI', 'Radar_VH', 'Elevation', 'Slope']
---------------------------------------------------


In [6]:
# ---------------------------------------------------------
# PHASE 4: NORMALIZATION & OBIA (SEGMENTATION)
# ---------------------------------------------------------

print("Starting Phase 4: Normalization & SNIC Segmentation...")


# ---------------------------------------------------------
# 4.1 Robust Scaling (Median / IQR)
# ---------------------------------------------------------

print("Computing robust scaling statistics...")

band_names = dna_stack_purified.bandNames()

stats = dna_stack_purified.reduceRegion(
    reducer = ee.Reducer.median().combine(
        reducer2 = ee.Reducer.percentile([25, 75]),
        sharedInputs = True
    ),
    geometry = roi,
    scale = 10,
    maxPixels = 1e9
)

print("Statistics computed.")


def scale_band(band):

    band = ee.String(band)

    median = ee.Number(stats.get(band.cat('_median')))
    p25 = ee.Number(stats.get(band.cat('_p25')))
    p75 = ee.Number(stats.get(band.cat('_p75')))

    iqr = p75.subtract(p25).max(0.00001)

    scaled = (
        dna_stack_purified
        .select(band)
        .subtract(median)
        .divide(iqr)
        .rename(band)
    )

    return scaled


scaled_list = band_names.map(scale_band)

dna_stack_scaled = ee.ImageCollection.fromImages(scaled_list).toBands()

# Remove auto prefixes from band names
dna_stack_scaled = dna_stack_scaled.rename(band_names)

print("Robust scaling applied.")


# ---------------------------------------------------------
# 4.2 SNIC Segmentation (Object Based Image Analysis)
# ---------------------------------------------------------

print("Running SNIC segmentation...")

snic = ee.Algorithms.Image.Segmentation.SNIC(
    image = dna_stack_scaled,
    size = 3,               # 30m seed spacing
    compactness = 0.1,
    connectivity = 8,
    neighborhoodSize = 6
)

# Keep only the mean values per segment
forest_stands = (
    snic
    .select(['.*_mean'], dna_stack_scaled.bandNames())
    .updateMask(forest_only_domain)
)

print("---------------------------------------------------")
print("PHASE 4 COMPLETE")
print("Data normalized and segmented into forest stands.")
print("Bands in segmented stack:")
print(forest_stands.bandNames().getInfo())
print("---------------------------------------------------")

Starting Phase 4: Normalization & SNIC Segmentation...
Computing robust scaling statistics...
Statistics computed.
Robust scaling applied.
Running SNIC segmentation...
---------------------------------------------------
PHASE 4 COMPLETE
Data normalized and segmented into forest stands.
Bands in segmented stack:
['Productivity', 'Seasonality', 'TCB', 'TCW', 'NDBI', 'Radar_VH', 'Elevation', 'Slope']
---------------------------------------------------


In [7]:
# ---------------------------------------------------------
# PHASE 5: AUTONOMOUS TAXONOMIC CLUSTERING (AUTO K)
# ---------------------------------------------------------

print("Starting Phase 5: Autonomous Clustering with Auto-K Selection...")


# ---------------------------------------------------------
# 5.1 Training Sample
# ---------------------------------------------------------

feature_bands = forest_stands.bandNames()

training_sample = forest_stands.sample(
    region = roi,
    scale = 10,
    numPixels = 5000,
    seed = 42,
    tileScale = 4
)

print("Training sample generated.")


# ---------------------------------------------------------
# 5.2 Test Multiple K Values
# ---------------------------------------------------------

print("Testing cluster counts (K = 2 → 8)...")

k_values = ee.List.sequence(2,8)

def evaluate_k(k):

    clusterer = ee.Clusterer.wekaKMeans(k).train(
        features = training_sample,
        inputProperties = feature_bands
    )

    clustered = training_sample.cluster(clusterer)

    # Compute variance for each feature band
    variance_dict = clustered.reduceColumns(
        reducer = ee.Reducer.variance().forEach(feature_bands),
        selectors = feature_bands
    )

    # Convert dictionary values to list and compute mean variance
    variance_values = ee.Dictionary(variance_dict).values()
    score = ee.Array(variance_values).reduce('mean', [0]).get([0])

    return ee.Feature(None, {
        'k': k,
        'score': score
    })

scores = ee.FeatureCollection(k_values.map(evaluate_k))

print("Cluster evaluation complete.")


# ---------------------------------------------------------
# 5.3 Select Best K
# ---------------------------------------------------------

# 1. Get the server-side object
best_k_server = scores.sort('score').first().get('k')

# 2. Force Google's servers to send the actual number to Python
best_k_client = best_k_server.getInfo()

# 3. Print the clean number
print(f"Best K determined automatically: {best_k_client}")

# 4. Save it as a server object for the next step to use
best_k = ee.Number(best_k_server)


# ---------------------------------------------------------
# 5.4 Train Final Clusterer
# ---------------------------------------------------------

clusterer = ee.Clusterer.wekaKMeans(best_k).train(
    features = training_sample,
    inputProperties = feature_bands
)

print("Final clusterer trained.")


# ---------------------------------------------------------
# 5.5 Apply Clusterer
# ---------------------------------------------------------

taxonomic_map = (
    forest_stands
    .cluster(clusterer)
    .rename('Forest_Type')
)

print("Initial clustering complete.")


# ---------------------------------------------------------
# 5.6 Spatial Cleaning
# ---------------------------------------------------------

final_taxonomic_map = (
    taxonomic_map
    .focalMode(
        radius = 1,
        kernelType = 'square',
        units = 'pixels'
    )
    .updateMask(forest_only_domain)
)

print("---------------------------------------------------")
print("PHASE 5 COMPLETE")
print("Cluster count selected automatically.")
print("Taxonomic map generated.")
print("---------------------------------------------------")

Starting Phase 5: Autonomous Clustering with Auto-K Selection...
Training sample generated.
Testing cluster counts (K = 2 → 8)...
Cluster evaluation complete.
Best K determined automatically: 2
Final clusterer trained.
Initial clustering complete.
---------------------------------------------------
PHASE 5 COMPLETE
Cluster count selected automatically.
Taxonomic map generated.
---------------------------------------------------


In [8]:
# ---------------------------------------------------------
# PHASE 6: ECOLOGICAL PROFILING (STABLE VERSION)
# ---------------------------------------------------------

print("Starting Phase 6: Ecological Profiling...")

# ---------------------------------------------------------
# 1. Convert clusters to vectors
# ---------------------------------------------------------

cluster_vectors = final_taxonomic_map.reduceToVectors(
    geometry = roi,
    scale = 10,
    geometryType = 'polygon',
    labelProperty = 'cluster',
    maxPixels = 1e13
)

print("Cluster polygons created.")

# ---------------------------------------------------------
# 2. Compute ecological statistics
# ---------------------------------------------------------

cluster_stats = dna_stack_raw.reduceRegions(
    collection = cluster_vectors,
    reducer = ee.Reducer.median(),
    scale = 10
)

print("Ecological statistics calculated.")

# ---------------------------------------------------------
# 3. Retrieve results
# ---------------------------------------------------------

results = cluster_stats.getInfo()

print("\n--- DNA PASSPORTS (VERIFIED) ---")

for feature in results['features']:
    print(feature['properties'])

# ---------------------------------------------------------
# 4. VISUALIZATION
# ---------------------------------------------------------

palette = [
'#00441b',
'#238b45',
'#74c476',
'#c7e9c0',
'#8c510a'
]

styled_map = final_taxonomic_map.visualize(
    min = 0,
    max = 4,
    palette = palette
)

black_background = ee.Image(0).visualize(palette=['black'])

final_figure = black_background.blend(styled_map).clip(roi)

print("---------------------------------------------------")
print("PIPELINE COMPLETE.")

Starting Phase 6: Ecological Profiling...
Cluster polygons created.
Ecological statistics calculated.

--- DNA PASSPORTS (VERIFIED) ---
{'Elevation': 242, 'NDBI': -0.060745279406850115, 'Productivity': 0.5571507215499878, 'Radar_VH': -14.41488758712698, 'Seasonality': 0.15565686509297114, 'Slope': None, 'TCB': 0.2681439909979701, 'TCW': -0.07661756917946039, 'cluster': 1, 'count': 1}
{'Elevation': 240, 'NDBI': -0.056813790849097046, 'Productivity': 0.5507140159606934, 'Radar_VH': -13.990447483975835, 'Seasonality': 0.18141788265014558, 'Slope': 2.304903745651245, 'TCB': 0.27794693001372117, 'TCW': -0.0696716892497614, 'cluster': 1, 'count': 186}
{'Elevation': 242, 'NDBI': -0.20806502907468594, 'Productivity': 0.6821533696992057, 'Radar_VH': -13.758621143666456, 'Seasonality': 0.11937132306172975, 'Slope': 2.9733458360036216, 'TCB': 0.2712402902661823, 'TCW': -0.03741851942405338, 'cluster': 0, 'count': 490}
---------------------------------------------------
PIPELINE COMPLETE.


In [9]:
import pandas as pd
import numpy as np

# 1. Convert your fragmented Earth Engine output into a Pandas DataFrame
features = [feat['properties'] for feat in results['features']]
df = pd.DataFrame(features)

# 2. Calculate the "Weighted Average" for each cluster based on patch size ('count')
def weighted_avg(group):
    d = group.drop(columns=['cluster', 'count'])
    w = group['count']
    return (d.multiply(w, axis=0).sum()) / w.sum()

# 3. Group by Cluster ID and apply the math
macro_passports = df.groupby('cluster').apply(weighted_avg).reset_index()
# macro_passports = df.groupby('cluster').apply(weighted_avg, include_groups=False).reset_index()

# 4. Calculate total area (in hectares) for each cluster
macro_passports['Area_Hectares'] = df.groupby('cluster')['count'].sum().values * 100 / 10000

print("\n--- FINAL TABLE (MACRO PASSPORTS) ---")
print(macro_passports.to_string(index=False))


--- FINAL TABLE (MACRO PASSPORTS) ---
 cluster  Elevation      NDBI  Productivity   Radar_VH  Seasonality    Slope      TCB       TCW  Area_Hectares
       0 242.000000 -0.208065      0.682153 -13.758621     0.119371 2.973346 0.271240 -0.037419           4.90
       1 240.010695 -0.056835      0.550748 -13.992717     0.181280 2.292578 0.277895 -0.069709           1.87


/tmp/ipykernel_3688/1368003895.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  macro_passports = df.groupby('cluster').apply(weighted_avg).reset_index()


In [20]:
import folium

# 1. Initialize the interactive map centered on Sanjay Van
lat, lon = 28.53, 77.175
m = folium.Map(location=[lat, lon], zoom_start=15, control_scale=True)

# 2. Add a High-Res Satellite Basemap (for real-world context underneath)
folium.TileLayer(
    tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
    attr='Google Satellite',
    name='Google Satellite Base',
    overlay=False,
    control=True
).add_to(m)

# 3. Add Layer 1: The Species Classification (The Forest)
# (Assuming your 2 main classes are 0: Kikar, 1: Native)
taxonomic_vis = final_taxonomic_map.getMapId({
    'min': 0,
    'max': 1,
    'palette': ['#b2df8a', '#006d2c'] # Light green for Kikar, Dark green for Native Core
})

folium.TileLayer(
    tiles=taxonomic_vis['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    name='🌲 Species Classification',
    overlay=True,
    control=True
).add_to(m)

# 4. Add Layer 2: The Blackout Mask (The Ignored Areas)
# We take the forest_only_domain, invert it (Not), and selfMask it
# so the forest areas become completely transparent, leaving only the black mask.
ignored_areas = forest_only_domain.Not().selfMask()

mask_vis = ignored_areas.getMapId({
    'palette': ['#000000'], # Pure Black
    'opacity': 0.85         # 85% opacity so you can barely see the city through it, or set to 1.0 for solid black
})

folium.TileLayer(
    tiles=mask_vis['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    name='⬛ Urban & Water Mask',
    overlay=True,
    control=True
).add_to(m)

# 5. Add the Layer Control (The Toggle Menu)
folium.LayerControl(position='topright').add_to(m)

# Display the map
m

In [12]:
import geemap
import ee

# 1. Define the Visual Styles (The "missing" variables)
forest_mask_vis = {'palette': ['000000'], 'opacity': 0.8}
cluster_vis = {'min': 0, 'max': 1, 'palette': ['#00441b', '#f1c40f']} # Dark Green vs Gold

# 2. Initialize the Map
Map = geemap.Map()

# 3. Add Background Layers (Toggles)
Map.add_basemap('SATELLITE')  # Google Satellite
Map.add_basemap('HYBRID')     # Google Satellite with Labels
Map.add_basemap('OpenStreetMap') # Standard OSM

# 4. Add the Blackout Layer (The City Mask)
Map.addLayer(forest_only_domain.clip(roi).Not(), forest_mask_vis, 'City Mask (Blackout)')

# 5. Add the AI Clusters
Map.addLayer(final_taxonomic_map.clip(roi), cluster_vis, 'AI Forest Taxonomy')

# 6. Add the Drone Spot (The Ground Truth)
Map.add_raster(tif_path, layer_name="Spot 1: Drone Footage")
Map.addLayer(spot_roi, {'color': 'red', 'fillColor': '00000000'}, 'Drone Footprint Border')

# 7. Final View Setup
Map.centerObject(spot_roi, 18) # Focus on the drone spot
Map.add_layer_control()

Map

Map(center=[28.532825079743517, 77.17672921536435], controls=(WidgetControl(options=['position', 'transparent_…

In [13]:
# 1. Create a 3-band visualization image (so colors are 'locked')
# We use .visualize() to turn the 0-1 clusters into actual RGB colors
# Then we .unmask(0) to turn the transparent city into solid BLACK
final_export_image = final_taxonomic_map.visualize(
    min=0,
    max=1,
    palette=['#00441b', '#f1c40f']
).unmask(0) # <--- THIS IS THE KEY: it fills the 'blackout' area with 0

# 2. Start the new Export
task = ee.batch.Export.image.toAsset(
    image = final_export_image,
    description = 'Sanjay_Van_Final_Baked_Map',
    assetId = f'projects/replicating-paper/assets/Sanjay_Van_Map_Baked',
    scale = 10,
    region = roi
)
task.start()

In [14]:
import time

while task.active():
    print(f"Task is {task.status()['state']}... checking again in 2s")
    time.sleep(2)

print(f"Final Status: {task.status()['state']}")

Task is READY... checking again in 2s
Task is READY... checking again in 2s
Task is READY... checking again in 2s
Final Status: FAILED


In [21]:
# Load the baked version
asset_id_baked = f'projects/replicating-paper/assets/Sanjay_Van_Map_Baked'
final_map_fixed = ee.Image(asset_id_baked)

Map = geemap.Map()
Map.add_basemap('SATELLITE')

# Add the baked image - No viz params needed!
Map.addLayer(final_map_fixed, {}, 'Final Baked Taxonomy (with Blackout)')

# Add drone footage
Map.add_raster(tif_path, layer_name="Spot 1: Drone Footage")

Map.centerObject(spot_roi, 18)
Map

Map(center=[28.532825079743517, 77.17672921536435], controls=(WidgetControl(options=['position', 'transparent_…